In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

(train_images, train_labels), (test_images, test_labels) = datasets.fashion_mnist.load_data()

train_images = train_images / 255.0
test_images  = test_images  / 255.0

train_images = train_images.reshape((-1, 28, 28, 1))
test_images  = test_images.reshape((-1, 28, 28, 1))

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu', name='last_conv'),  # named for Grad-CAM
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=10,
                    validation_data=(test_images, test_labels))

test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print("\nTest accuracy:", test_acc)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.tight_layout()
plt.show()

y_pred         = model.predict(test_images)
y_pred_classes = np.argmax(y_pred, axis=1)

cm   = confusion_matrix(test_labels, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(xticks_rotation=45, cmap='Blues')
plt.tight_layout()
plt.show()

print("\nClassification Report:\n")
print(classification_report(test_labels, y_pred_classes, target_names=class_names))

In [ ]:
import random

num_samples    = 10
random_indices = random.sample(range(len(test_images)), num_samples)

plt.figure(figsize=(15,5))

for i, idx in enumerate(random_indices):
    image      = test_images[idx]
    true_label = test_labels[idx]

    prediction     = model.predict(np.expand_dims(image, axis=0))
    predicted_label = np.argmax(prediction)

    plt.subplot(2, 5, i + 1)
    plt.imshow(image.reshape(28, 28), cmap='gray')
    plt.title(f"True: {class_names[true_label]}\nPred: {class_names[predicted_label]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Grad-CAM Explainability ─────────────────────────────────────────────────
#
# Grad-CAM (Gradient-weighted Class Activation Mapping) answers:
# "Which pixels drove the model's prediction?"
#
# How it works:
#   1. Run a forward pass and record activations at the last conv layer.
#   2. Compute the gradient of the predicted class score w.r.t. those activations.
#   3. Global-average-pool the gradients → per-channel importance weights.
#   4. Weighted sum of activation maps → heatmap. Regions with high positive
#      values are what the model "looked at" most when making its decision.
# ─────────────────────────────────────────────────────────────────────────────

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random

# Build a sub-model that outputs both the last conv layer AND the final predictions
grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[model.get_layer('last_conv').output, model.output]
)

def compute_gradcam(image, class_index=None):
    """
    Returns a Grad-CAM heatmap for `image` (shape: 28x28x1).
    If class_index is None, uses the predicted class.
    """
    img_tensor = tf.cast(np.expand_dims(image, axis=0), tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        tape.watch(conv_outputs)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        class_score = predictions[:, class_index]

    # Gradients of the class score w.r.t. the conv feature maps
    grads = tape.gradient(class_score, conv_outputs)

    # Global average pooling over spatial dims → shape: (num_filters,)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight each filter's activation map by its importance
    conv_outputs = conv_outputs[0]                         # (H, W, filters)
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis] # (H, W, 1)
    heatmap = tf.squeeze(heatmap)

    # ReLU: keep only positive influence, then normalise to [0, 1]
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(image, heatmap, alpha=0.5):
    """
    Upsamples heatmap to image size and overlays it as a coloured mask.
    Returns an RGB array ready for imshow.
    """
    # Upsample heatmap from conv output size (5x5) → image size (28x28)
    heatmap_resized = np.array(
        tf.image.resize(heatmap[..., np.newaxis], (28, 28))
    ).squeeze()

    # Apply a colour map
    colormap   = cm.get_cmap('jet')
    heatmap_rgb = colormap(heatmap_resized)[:, :, :3]  # drop alpha channel

    # Normalise original image to [0,1] for RGB blending
    img_rgb = np.stack([image.squeeze()] * 3, axis=-1)
    img_rgb = img_rgb / img_rgb.max()

    # Blend
    overlay = (1 - alpha) * img_rgb + alpha * heatmap_rgb
    overlay = np.clip(overlay, 0, 1)
    return overlay

# ── Visualise Grad-CAM for 10 random test images ─────────────────────────────
num_samples    = 10
random_indices = random.sample(range(len(test_images)), num_samples)

fig, axes = plt.subplots(3, num_samples, figsize=(20, 6))
fig.suptitle("Grad-CAM Explainability — Fashion MNIST CNN", fontsize=14, y=1.02)

for col, idx in enumerate(random_indices):
    image      = test_images[idx]
    true_label = test_labels[idx]

    pred_probs      = model.predict(np.expand_dims(image, axis=0), verbose=0)
    predicted_label = np.argmax(pred_probs)
    confidence      = pred_probs[0][predicted_label] * 100

    heatmap = compute_gradcam(image, class_index=predicted_label)
    overlay = overlay_gradcam(image, heatmap, alpha=0.55)

    # Row 0: original image
    axes[0, col].imshow(image.reshape(28, 28), cmap='gray')
    axes[0, col].set_title(f"True:\n{class_names[true_label]}", fontsize=7)
    axes[0, col].axis('off')

    # Row 1: raw Grad-CAM heatmap
    axes[1, col].imshow(heatmap, cmap='jet')
    axes[1, col].set_title(f"Pred: {class_names[predicted_label]}\n{confidence:.1f}%", fontsize=7,
                            color='green' if predicted_label == true_label else 'red')
    axes[1, col].axis('off')

    # Row 2: overlay on original
    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Overlay", fontsize=7)
    axes[2, col].axis('off')

# Column labels
axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel("Heatmap", fontsize=9)
axes[2, 0].set_ylabel("Grad-CAM\nOverlay", fontsize=9)

plt.tight_layout()
plt.show()

# ── Per-class Grad-CAM: show what the model looks for in each class ───────────
print("\n--- Per-class Grad-CAM: one correctly-predicted sample per class ---\n")

fig2, axes2 = plt.subplots(3, 10, figsize=(20, 6))
fig2.suptitle("Per-class Grad-CAM — what the model attends to per category",
               fontsize=12, y=1.02)

for cls_idx in range(10):
    # Find a correctly-predicted sample for this class
    candidates = np.where(
        (test_labels == cls_idx) & (y_pred_classes == cls_idx)
    )[0]
    if len(candidates) == 0:
        for row in range(3):
            axes2[row, cls_idx].axis('off')
        continue

    idx    = candidates[0]
    image  = test_images[idx]

    heatmap = compute_gradcam(image, class_index=cls_idx)
    overlay = overlay_gradcam(image, heatmap, alpha=0.55)

    axes2[0, cls_idx].imshow(image.reshape(28, 28), cmap='gray')
    axes2[0, cls_idx].set_title(class_names[cls_idx], fontsize=8)
    axes2[0, cls_idx].axis('off')

    axes2[1, cls_idx].imshow(heatmap, cmap='jet')
    axes2[1, cls_idx].axis('off')

    axes2[2, cls_idx].imshow(overlay)
    axes2[2, cls_idx].axis('off')

axes2[0, 0].set_ylabel("Original", fontsize=9)
axes2[1, 0].set_ylabel("Heatmap", fontsize=9)
axes2[2, 0].set_ylabel("Overlay", fontsize=9)

plt.tight_layout()
plt.show()

print("Grad-CAM complete.")
print("Hot (red/yellow) regions = pixels the model weighted most for its prediction.")
print("Cool (blue) regions = low influence on the decision.")